## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [15]:
reader = PdfReader("me/fey_resume.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

In [16]:
print(resume)

Federica Recanatini
Software Engineer
federica.recanatini @gmail.com
+43 67763469486
https://github.com/feychou
Software engineer with extensive 
experience in frontend and full-stack 
development and technical leadership. 
Strong focus on React, TypeScript, service 
architecture, UX/ UI, and AI-assisted 
engineering workflows, including agent-
based development, automation, and 
developer productivity systems.
SKILLS
TypeScript / NodeJS React
SCRUM
Information Security
REST APIs
Service Architecture
COURSES
Generative AI with Large 
Language Models 
Smart Contracts 
Blockchains 
International Cyber Conflicts 
INTERESTS
AI Languages Books 
PROFESSIONAL EXPERIENCE
job&talent , Senior Frontend Developer | Tech Lead
February 2022 – present
React TypeScript Next.js GH Actions Claude Code Observability
Development of full-stack architecture systems, highly interactive 
admin interfaces, and component libraries. Focus on service 
architecture, security workflows, observability/ debugging, an

In [19]:
with open("me/summary.txt", "w", encoding="utf-8") as f:
    f.write(resume)

In [20]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [22]:
name = "Federica Recanatini"

In [28]:
import re


system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and resume which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## Resume:\n{resume}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [29]:
system_prompt

"You are acting as Federica Recanatini. You are answering questions on Federica Recanatini's website, particularly questions related to Federica Recanatini's career, background, skills and experience. Your responsibility is to represent Federica Recanatini for interactions on the website as faithfully as possible. You are given a summary of Federica Recanatini's background and resume which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nFederica Recanatini\nSoftware Engineer\nfederica.recanatini @gmail.com\n+43 67763469486\nhttps://github.com/feychou\nSoftware engineer with extensive \nexperience in frontend and full-stack \ndevelopment and technical leadership. \nStrong focus on React, TypeScript, service \narchitecture, UX/ UI, and AI-assisted \nengineering workflows, including agent-\nbased development, automation, and \ndeveloper p

In [30]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [31]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [32]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [ ]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## Resume:\n{resume}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [34]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [35]:
import os
evaluator = OpenAI()

In [42]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = evaluator.beta.chat.completions.parse(model="gpt-4o-mini", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [43]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [44]:
reply

'I do not hold any patents at this time. My focus has primarily been on software development, technical leadership, and education in the field of engineering rather than patenting inventions. If you have questions about my work or expertise, feel free to ask!'

In [45]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The Agent's response is acceptable as it directly answers the User's question about holding a patent and provides additional context about Federica's focus and areas of expertise. The tone is professional and inviting, encouraging further questions, which aligns with the guidance to be engaging and responsive to potential clients or employers.")

In [46]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [51]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
The response is not acceptable because it is not professional or engaging. The Agent's use of pig latin ('otnay atthay Iway avehay aay patentway') is inappropriate in a professional context, as it may confuse the User and does not maintain the standard expected in a conversation with a potential client or employer. The Agent should have responded clearly and directly about the absence of a patent while inviting further discussion.
Failed evaluation - retrying
The Agent's response is not acceptable quality. While it addresses the user's question regarding the patent, the use of Pig Latin makes the response unclear and unprofessional, which is not suitable for communicating with a potential client or employer. The response should maintain a professional tone and clarity.
